코드 thesis-method-section

In [ ]:
## 3.4 분석 절차 — LLM을 활용한 자동 코딩

### 3.4.1 사용 모델과 도구

본 연구의 5요소 프레임 분석에는 Anthropic의 Claude Opus 4.7
(claude-opus-4-7, API 호출 일자: 2026.05.01~2026.05.15)을 주 모델로
사용하였다. 모델 선택은 다음 세 가지 기준에 따라 이루어졌다.
첫째, 한국어 능력 — Claude Opus 4.7은 한국어 GPQA 벤치마크에서
경쟁 모델 대비 0.05~0.08 높은 정확도를 보였다. 둘째, 추론 능력 —
Matthes & Kohring(2008)의 5요소 추출은 다단계 추론을 요구하며, Opus
계열은 이 작업에 강점을 갖는다. 셋째, 환각률 — Artificial Analysis
2026년 4월 측정 기준 Opus 4.7의 환각률(36%)이 동급 모델 중 가장 낮다.

비교 검증 목적으로 GPT-5.5와 HyperCLOVA X의 동일 작업 결과도 산출하여
부록 C에 제시하였다.

### 3.4.2 시스템 프롬프트와 호출 매개변수

시스템 프롬프트는 Matthes & Kohring(2008)의 5요소 정의에 기반하여
설계되었으며, 전문은 부록 A에 수록되었다. 핵심 설계 원칙은 다음과 같다.
(1) 각 요소(문제·원인·도덕·해결·행위자)의 정확한 정의 명시,
(2) ‘분류’가 아닌 ‘추출’의 강조(인덕티브성 보존),
(3) 구조화된 JSON 출력의 강제,
(4) 신뢰도 메타데이터(confidence, text_basis)의 요청,
(5) ‘불확실하면 unclear/none을 사용하라’의 환각 방지 지시.

호출 매개변수는 다음과 같다.
temperature=0 (재현가능성을 위해),
max_tokens=1500, top_p=1.0,
response_format="json_object".

### 3.4.3 신뢰도 검증

자동 코딩의 신뢰도는 다음 절차로 검증되었다.
첫째, 전체 데이터의 약 2.3%인 200건을 무작위 추출.
둘째, 두 명의 훈련된 인간 코더(박사과정생 1인, 석사과정생 1인)가
동일한 200건에 대해 독립적으로 5요소를 추출.
셋째, 코더 간 및 LLM-코더 간 Krippendorff's α를 계산.

결과: 코더 간 α = 0.78 (문제), 0.72 (원인), 0.81 (도덕),
0.69 (해결), 0.85 (행위자). LLM-코더 평균 α = 0.74, 0.68, 0.76,
0.63, 0.82. LLM의 두 번 실행 일관성 α = 0.94 (모든 차원).

이 결과는 LLM 자동 코딩이 인간 코더 수준의 90~95%에 도달함을 시사한다.
해결책 권고(solution) 차원에서 α가 가장 낮은 것은 인간 코더 간에도
공통된 어려움이었으며, 이 차원의 결과는 신중히 해석하였다.

### 3.4.4 윤리적 고려

본 연구의 분석 대상은 공개된 한국 주요 일간지 기사로, IRB 면제 대상이다
(서울대학교 IRB 사전 면제 확인서: IRB-2026-XX-XXXX).
LLM API 호출 시 기사 본문은 외부 서버로 전송되었으나,
Anthropic의 정책상 API 호출 데이터는 학습에 사용되지 않으며
30일 후 자동 삭제된다(Anthropic, 2024). 다만 본 연구에서는
신중을 기하여, 기사 본문에 포함된 개인 식별 정보(일반 시민의 실명·
주소 등)는 사전에 [PERSON_A] 등으로 마스킹한 후 호출하였다.

### 3.4.5 공개 자료

본 연구의 다음 자료는 OSF(github.com/kweon/dissertation)에
공개되었다. 1) 시스템 프롬프트 전문, 2) 분석 코드, 3) 인간 코더 검증
샘플 200건의 라벨, 4) 8,800건 분석 결과의 집계 데이터(article_id +
5요소 라벨 + confidence). 기사 본문은 저작권 문제로 공개되지 않으나,
각 article_id에 해당하는 URL이 함께 제공된다.

코드 prompt-journal

In [ ]:
# 프롬프트 일지 — [프로젝트명]

## 2026-05-10 14:30 | 모델: claude-opus-4-7

### 시스템 프롬프트
없음 (일반 대화)

### 사용자 질문
"3장 결과 절에서 LLM의 정파성 분류 결과를 어떻게 시각화할까?
변수: 매체별·시기별 정파성 비율, 8,800건."

### LLM 응답 (요약)
3가지 제안:
1. 매체별 박스 플롯 (5개 매체 × 4 정파성 카테고리)
2. 시기별 적층 막대 그래프 (월별, 2020~2024)
3. 매체-시기 히트맵 (5×60 격자)

### 내 평가
- 옵션 3 채택. 다만 LLM이 ‘5×60’의 격자가 시각적으로 너무
  세밀하다는 것은 지적하지 않음. 인간이 ‘5×20 (분기별)’로
  단순화하기로 결정.
- 옵션 1·2는 부록에 보조로 포함.

### 최종 반영
- 본문 그림 3.1: 매체-시기 히트맵 (5×20). 1차 결과는
  Python matplotlib으로 직접 작성, 코드는 GitHub 공개.
- 부록 D.2: 매체별 박스 플롯 (옵션 1).

## 2026-05-11 09:15 | 모델: claude-opus-4-7
...